# Phase 4 — Degree-only GCN with the accepted graph protocol

This is a protocol and reproducibility refactor of the original GCN experiment. The original notebook and historical outputs are preserved in `archive/04_graph_baseline_pre_v2.ipynb` and the legacy artifact directories.

## 1. Objective and preserved methodology

- GCNConv 3 → 64 → 32; ReLU after both convolutions; dropout 0.3 after the first; linear 32 → 2.
- Raw degree inputs computed after appending reversed edges: `[d, d, 2d]`. These three columns are redundant and remain unchanged in this first pass.
- Reverse-edge concatenation retains multiplicity when an observed relation is reciprocal. No silent coalescing or degree scaling.
- Weighted cross-entropy, AdamW lr=0.01 and weight_decay=0.0005, full-batch training for 200 epochs.
- Select the lowest validation-loss checkpoint; preserve the final checkpoint separately. Bot probability threshold remains 0.5.

Corrections: use three separate permitted Phase-2 graphs, training-only class weights, canonical user keys, deterministic execution, deep-copied checkpoint states, explicit missing-graph users, and versioned outputs. These change the experimental protocol and cohort, not the architecture; historical scores are not directly comparable.

**Protocol:** held-out gold target users, with shared support accounts. This is not a fully node-disjoint experiment. Validation/test users are evaluated jointly on their respective cohort graph; this is not isolated one-account deployment.

## 2. Configuration, manifest and reproducibility

Select `configs/graph_baseline_v2.json` explicitly. Input paths/hashes are pinned, never selected by a newest-directory search. Run from the project root or `analysis/` using `bot_detection_env`.

No raw-data parsing, preprocessing fitting, split regeneration, text embedding generation or unlabeled-owner graph augmentation occurs here. Runtime implementation is in `src/graph_baseline.py`; generic artifact/metric helpers are reused from Phase 3 without modifying them.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / 'configs/graph_baseline_v2.json').is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.graph_baseline import GraphBaselineRun, GCNClassifier, training_class_weights, verify_graph_run

run = GraphBaselineRun(PROJECT_ROOT)
print('Phase-2 manifest:', run.config['phase2_manifest'])
print('SHA256:', run.config['phase2_manifest_sha256'])
print('Output:', run.output_dir)
print('Device:', run.config['device'], '| seed:', run.config['seed'])
display(pd.Series(run.provenance['packages'], name='version').to_frame())

C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\env\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Phase-2 manifest: final_data/versions/preprocessing_v2_e861084d1a722a21/manifest.json
SHA256: 594a2defef59d663ef171020b7cd1ba75aa321c30e7a7d09b57b560563c109c9
Output: C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\results\versions\graph_baseline_v2\20260916T113841559166Z_2dd11f1e
Device: cpu | seed: 42


,version
torch,2.14.0
torch-geometric,2.8.0.post1
numpy,2.4.6
pandas,3.0.5
scikit-learn,1.9.0


## 3. Canonical populations and graph coverage

`user_key` is the only identity key. Graph node indices are local tensor positions, not account identifiers. Numeric Twitter IDs are not model inputs or join keys.

Primary eligibility follows the Phase-2 `has_graph` mask: owner-neighborhood evidence plus a retained incident edge. A user can occur as a graph node while lacking this owner evidence. Such users may provide observed connectivity but are not supervised/evaluated targets in this experiment.

All canonical users remain in prediction tables. Unsupported graph users have null probability/prediction and a missing-graph status, with node-only evidence distinguished from absence of a permitted node. No invented neighborhoods, fake graph vectors, or deletion due to missing text/temporal modalities.

In [2]:
display(run.coverage)
for split, graph in run.graphs.items():
    masks = graph['masks']
    print(split, 'canonical users:', len(graph['users']),
          '| graph-eligible:', int(masks.has_graph.sum()),
          '| missing graph:', int(masks.has_graph.eq(0).sum()))

,split,all_users,graph_available,eligible_humans,eligible_bots,node_only_users,nodes,directed_observed_edges,message_passing_entries,repeated_message_entries,forbidden_gold_nodes
0,train,671,193,162,31,380,42954,60924,121848,3964,0
1,validation,144,42,35,7,42,11436,13006,26012,336,0
2,test,144,41,35,6,48,10453,12536,25072,1210,0


train canonical users: 671 | graph-eligible: 193 | missing graph: 478
validation canonical users: 144 | graph-eligible: 42 | missing graph: 102
test canonical users: 144 | graph-eligible: 41 | missing graph: 103


## 4. Edge integrity, collection-owner provenance and split isolation

Before tensor construction the loader verifies:

1. Every required artifact SHA256 against the selected manifest.
2. Frozen user keys and labels; binary modality masks agreeing with canonical users.
3. Canonical edge keys; one `follows` relation; no observed self-edges or duplicate directed relations.
4. Edge-owner rows exactly explain the permitted edges; each owner belongs to this split and is an endpoint.
5. No gold endpoint from another split. Node membership and `has_graph` agree with actual evidence.
6. Graph counts match the Phase-2 audit report.

The global audit graph and legacy `internal_edge` flags are never model inputs. Support endpoints outside the canonical user table remain valid observed nodes: they need neither labels nor tabular/text features for this degree-only GCN.

In [3]:
display(run.support_overlap)
assert run.coverage.forbidden_gold_nodes.eq(0).all()
print('Input hashes, edge provenance, split membership and mask checks passed.')

,left,right,shared_support_nodes
0,train,validation,4035
1,train,test,3919
2,validation,test,1660


Input hashes, edge provenance, split membership and mask checks passed.


## 5. Separate graphs and deterministic mappings

Training uses only the training graph. Validation and test have separate tensors and sorted `user_key` mappings. Shared support accounts do not share cached hidden representations across graphs. Each forward pass computes from its own graph.

`GCNConv(cached=False)` is explicit. Standard GCN self-loops are computational operations; they are not claimed as observed Twitter relationships. The saved graph edges contain only original permitted relations and their computational reverses, before GCN self-loops.

In [4]:
for split, graph in run.graphs.items():
    print(split, graph['data'])
    display(graph['mapping'].head())

train Data(x=[42954, 3], edge_index=[2, 121848], y=[42954], eligible_mask=[42954])


,user_key,node_index
0,000terminator,0
1,002211_01,1
2,002_milad,2
3,007mi00,3
4,0098_santiago,4


validation Data(x=[11436, 3], edge_index=[2, 26012], y=[11436], eligible_mask=[11436])


,user_key,node_index
0,007ramez35821,0
1,00chsaaraa,1
2,010101abba,2
3,0631arash301,3
4,06deadline,4


test Data(x=[10453, 3], edge_index=[2, 25072], y=[10453], eligible_mask=[10453])


,user_key,node_index
0,000terminator,0
1,002211_01,1
2,01niman,2
3,0d5823x5,3
4,0ldlo,4


## 6. Degree-feature and edge-multiplicity diagnostics

Features count the entries in this split's symmetrized edge tensor before computational self-loops. They are partial observed-graph degrees, not complete Twitter follower/following counts. Reciprocal directed observations contribute repeated message connections; this historical weighting convention is preserved explicitly.

No global degrees, scaling fitted on held-out nodes, or additional structural features are introduced. Revising this redundant degree representation is a later experiment.

In [5]:
import torch
for split, graph in run.graphs.items():
    x = graph['data'].x
    assert torch.equal(x[:, 0], x[:, 1])
    assert torch.equal(x[:, 2], 2 * x[:, 0])
    print(split, '| feature shape:', tuple(x.shape), '| maxima:', x.max(dim=0).values.tolist())
display(run.coverage[['split', 'directed_observed_edges', 'message_passing_entries', 'repeated_message_entries']])

train | feature shape: (42954, 3) | maxima: [389.0, 389.0, 778.0]
validation | feature shape: (11436, 3) | maxima: [360.0, 360.0, 720.0]
test | feature shape: (10453, 3) | maxima: [375.0, 375.0, 750.0]


,split,directed_observed_edges,message_passing_entries,repeated_message_entries
0,train,60924,121848,3964
1,validation,13006,26012,336
2,test,12536,25072,1210


## 7. GCN and training-only class weights

The original balanced weight formula is retained: `N_train / (2 * count_class_train)`. Only eligible training users contribute. Support nodes and graph-unavailable gold users have label slots of -1 and are excluded from loss.

Validation labels are used to select the checkpoint, never to set training class weights. Test labels do not enter training or checkpoint selection. The training function accepts training and validation graph objects only.

In [6]:
print(GCNClassifier(**run.config['architecture']))
weights, counts = training_class_weights(run.graphs['train']['data'])
print('Training class counts [human, bot]:', counts.tolist())
print('Training class weights:', weights.tolist())
print('Training settings:', run.config['training'])

GCNClassifier(
  (conv1): GCNConv(3, 64)
  (conv2): GCNConv(64, 32)
  (classifier): Linear(in_features=32, out_features=2, bias=True)
)
Training class counts [human, bot]: [162, 31]
Training class weights: [0.595678985118866, 3.112903118133545]
Training settings: {'epochs': 200, 'lr': 0.01, 'weight_decay': 0.0005, 'selection': 'best_validation_loss', 'threshold': 0.5}


## 8. Train and save independent best/final checkpoints

The original `.state_dict().copy()` was shallow, so later optimization mutated the purported best weights. The refactor uses `deepcopy`. Train the full 200 epochs with no early stopping; save both best-validation and final snapshots plus the complete history.

In [7]:
history = run.fit()
print('Selected best-validation epoch:', run.info['best_epoch'])
print('Final epoch:', run.info['last_epoch'])
display(history.tail())

Epoch 20/200: train=1.804969, validation=0.804632


Epoch 40/200: train=0.698299, validation=0.764608


Epoch 60/200: train=0.718676, validation=0.695139


Epoch 80/200: train=0.696431, validation=0.711353


Epoch 100/200: train=0.708839, validation=0.694679


...

Epoch 160/200: train=0.687964, validation=0.695566


Epoch 180/200: train=0.667998, validation=0.697488


Epoch 200/200: train=0.678148, validation=0.695629


Selected best-validation epoch: 56
Final epoch: 200


,epoch,train_loss,validation_loss
195,196,0.690203,0.696221
196,197,0.683270,0.696328
197,198,0.700774,0.695666
198,199,0.673597,0.695583
199,200,0.678148,0.695629


## 9. Keyed evaluation and matched baseline comparisons

Save all canonical users for every checkpoint/split. Scores on `all_users` describe the supported subset, with the full population denominator and coverage explicit. `graph_available` is the primary graph-supported cohort.

Report bot precision, recall, F1, ROC-AUC, average precision, accuracy, class counts, confusion matrices and classification reports. Training scores are apparent fit performance. The graph-supported test cohort has very few bots; a single run does not estimate uncertainty.

Compare the selected GCN with the four saved Phase-3 primary baselines only on the common supported keys. The exact comparison keys are exported. Models are not retrained or selected using these comparisons, and test results do not choose a checkpoint.

In [8]:
metrics = run.evaluate_and_export()
display(metrics[metrics.primary_checkpoint & metrics.split.isin(['validation', 'test'])])
display(run.comparison)
print('Complete output manifest:', run.output_dir / 'manifest.json')

,model,checkpoint_variant,primary_checkpoint,split,cohort,population_n,evaluated_n,missing_predictions_n,coverage,human_n,bot_n,accuracy,bot_precision,bot_recall,bot_f1,roc_auc,average_precision
2,gcn_degree_only,best_validation_loss,True,validation,all_users,144,42,102,0.291667,35,7,0.142857,0.128205,0.714286,0.217391,0.477551,0.244015
3,gcn_degree_only,best_validation_loss,True,validation,graph_available,42,42,0,1.000000,35,7,0.142857,0.128205,0.714286,0.217391,0.477551,0.244015
4,gcn_degree_only,best_validation_loss,True,test,all_users,144,41,103,0.284722,35,6,0.243902,0.162162,1.000000,0.279070,0.595238,0.238547
5,gcn_degree_only,best_validation_loss,True,test,graph_available,41,41,0,1.000000,35,6,0.243902,0.162162,1.000000,0.279070,0.595238,0.238547


,model,split,cohort,population_n,evaluated_n,missing_predictions_n,coverage,human_n,bot_n,accuracy,bot_precision,bot_recall,bot_f1,roc_auc,average_precision
0,gcn_degree_only,validation,graph_available_and_all_baselines_supported,42,42,0,1.0,35,7,0.142857,0.128205,0.714286,0.217391,0.477551,0.244015
1,tabular_lr,validation,graph_available_and_all_baselines_supported,42,42,0,1.0,35,7,0.833333,0.500000,0.714286,0.588235,0.861224,0.578175
2,tabular_mlp,validation,graph_available_and_all_baselines_supported,42,42,0,1.0,35,7,0.690476,0.285714,0.571429,0.380952,0.738776,0.558929
3,text_lr,validation,graph_available_and_all_baselines_supported,42,42,0,1.0,35,7,0.666667,0.333333,1.000000,0.500000,0.869388,0.556181
4,text_mlp,validation,graph_available_and_all_baselines_supported,42,42,0,1.0,35,7,0.404762,0.218750,1.000000,0.358974,0.865306,0.603194
5,gcn_degree_only,test,graph_available_and_all_baselines_supported,40,40,0,1.0,34,6,0.250000,0.166667,1.000000,0.285714,0.593137,0.241618
6,tabular_lr,test,graph_available_and_all_baselines_supported,40,40,0,1.0,34,6,0.725000,0.307692,0.666667,0.421053,0.848039,0.689712
7,tabular_mlp,test,graph_available_and_all_baselines_supported,40,40,0,1.0,34,6,0.725000,0.272727,0.500000,0.352941,0.799020,0.548134
8,text_lr,test,graph_available_and_all_baselines_supported,40,40,0,1.0,34,6,0.650000,0.250000,0.666667,0.363636,0.769608,0.504118
9,text_mlp,test,graph_available_and_all_baselines_supported,40,40,0,1.0,34,6,0.450000,0.214286,1.000000,0.352941,0.754902,0.380981


Complete output manifest: C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\results\versions\graph_baseline_v2\20260916T113841559166Z_2dd11f1e\manifest.json


## 10. Reload verification and downstream handoff

Reconstruct all three graphs from pinned Phase-2 inputs and compare against saved tensors/mappings. Independently reload both model snapshots, reproduce their recorded validation losses and keyed predictions, and recompute every metric and matched comparison.

The run manifest records split-specific graph artifacts and checkpoint selection. Notebooks 05–07 still consume historical global graph artifacts and will be migrated individually; this notebook does not overwrite those files. Never merge these graph objects or reuse a node index across split mappings.

Limits remain: incomplete graph collection, redundant degree features, shared support nodes, small graph-available cohorts, and previously inspected historical test data. GCN computational reversals/self-loops do not imply newly observed social relationships.

In [9]:
verification = verify_graph_run(PROJECT_ROOT, run.output_dir)
print(verification)
print('Ready for Notebook 05 inspection; no downstream notebook was changed.')

{'verified_artifacts': 28, 'reloaded_checkpoints': 2, 'reconstructed_graphs': 3, 'verified_prediction_files': 6, 'verified_metric_rows': 12, 'matched_comparison_rows': 10}
Ready for Notebook 05 inspection; no downstream notebook was changed.
